## Single-value Repair Analyzer

In [ ]:
import pandas as pd

# You can read the already calculated repairs file, or read the one generated on 't-box repairs analyzer' folder with the script 't-box_repairs_analyzer.py'
df_single_val_repairs = pd.read_csv("../../datasets/single_value_repairs.csv")

df_single_val_repairs

- The cell below counts different types of basic T-box repairs generated with the relational database:

In [ ]:
print(len(df_single_val_repairs[(df_single_val_repairs['C_deleted'] == True)] ))
print(len(df_single_val_repairs[(df_single_val_repairs['C_deprecated'] == True)] ))
print(len(df_single_val_repairs[(df_single_val_repairs['CQ_added_exception'] == True)] ))

- Other columns of this file include: 
    - wds1 - (SQ) base statement qualifiers id
    - CQ_add_property_on_CQ_with_0 - True if constraint had no separators and later on got them, otherwise False (constraint remained with 0 separators)
    - no_separator - True if the constraint has no separators, otherwise False

- checking for deleted base statements:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def statementDeleted(obj_statement):
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""ASK {{ <{obj_statement}> ?p ?o  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'false'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
obj_statement = "http://www.wikidata.org/entity/statement/Q1001-6408985C-EAAD-4193-95F7-FCF2B179E6E3"
print(statementDeleted(obj_statement))

In [ ]:
df_single_val_repairs["S_deleted"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

# Load the data, or resume from the last checkpoint
checkpoint_file = "checkpoint.csv"
if os.path.exists(checkpoint_file):
    df_single_val_repairs = pd.read_csv(checkpoint_file)
    print("Resuming from the last checkpoint.")
else:
    print("Starting from scratch.")

# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['S_deleted']):
        result = statementDeleted(row['wds1'])
        df_single_val_repairs.at[index, 'S_deleted'] = result

    # Save a checkpoint every 10,000 rows
    if index % 10000 == 0:
        df_single_val_repairs.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
len(df_single_val_repairs[(df_single_val_repairs['S_deleted'] == True)] )

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

- listing all not yet classified rows:

In [ ]:
df_single_val_repairs[(df_single_val_repairs['C_deleted'] == False) & 
     (df_single_val_repairs['C_deprecated'] == False)& 
     (df_single_val_repairs['CQ_added_exception'] == False)& 
     (df_single_val_repairs['S_deleted'] == False) 
     
    ]

- checking for context statement (Sc) deletion:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def complementaryStatementDeleted(subject, wd_pid):
    
    p_pid = wd_pid.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    # URL of the endpoint
    endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""SELECT (COUNT( DISTINCT(?o)) AS ?total)
        WHERE
        {{
          <{subject}> <{p_pid}> ?o
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find the 'literal' element containing the 'total' value
        literal_element = root.find('.//ns:literal', namespace)

        # Extract the text from the 'literal' element and convert to an integer
        total_value = int(literal_element.text) if literal_element is not None else 0

        # Return True if total is 1, False otherwise
        result = total_value == 1
        
        return result
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

# Example usage
subject = "http://www.wikidata.org/entity/Q1048"
wd_pid = "http://www.wikidata.org/entity/P1003"
print(complementaryStatementDeleted(subject, wd_pid))


In [ ]:
df_single_val_repairs["Sc_deleted"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os

for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row['Sc_deleted']):
        if row['S_deleted'] is True:
            df_single_val_repairs.at[index, 'Sc_deleted'] = False
        else:      
            result = complementaryStatementDeleted(row['subject'], row['property'])
            df_single_val_repairs.at[index, 'Sc_deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["Sc_deleted"].value_counts()

- checking non-classified (yet) rows:

In [ ]:
df_single_val_repairs[(df_single_val_repairs['S_deleted'] == False) & 
     (df_single_val_repairs['no_separator'] == False )& 
     (df_single_val_repairs['CQ_add_property_on_CQ_with_0'] == False)#& 
     #(df_single_val_repairs['Sc_deleted'] == False)
]

In [ ]:
df_single_val_repairs.iloc[236962]['wds1']

In [ ]:
df_single_val_repairs.iloc[236962]

- Test for base statement qualifiers separator added (SQ+):

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getConstraintSeparator(wd_pid, endpoint = "ENTER_qEndpoint_WD_2023"):
    
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT ?separator
        WHERE
        {{
          <{wd_pid}> <http://www.wikidata.org/prop/P2302> ?constraint.
          ?constraint <http://www.wikidata.org/prop/statement/P2302> <http://www.wikidata.org/entity/Q19474404>.
          ?constraint <http://www.wikidata.org/prop/qualifier/P4155>/wikibase:qualifier ?separator.

          FILTER NOT EXISTS {{?constraint <http://www.wikidata.org/prop/qualifier/P2241> []}}
          FILTER NOT EXISTS {{?constraint wikibase:rank wikibase:DeprecatedRank}}
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return []
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

def hasSeparator(triple_statement, separator, endpoint):
    # SPARQL query
    query = f"""ASK {{ <{triple_statement}> <{separator}> ?o  }}
            """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)
        boolean_element = root.find('.//{http://www.w3.org/2005/sparql-results#}boolean')
        if boolean_element is not None:
            #print(boolean_element.text)
            return boolean_element.text.lower() == 'true'
        else:
            print("Error: 'boolean' element not found in XML response")
            return None
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    

def wasSeparatorValueAdded(wd_pid, triple_statement):
    
    # URL of the endpoint
    endpoint_19 = "ENTER_qEndpoint_WD_2019"
    endpoint_23 = "ENTER_qEndpoint_WD_2023"

    separators = getConstraintSeparator(wd_pid)
    if separators:
        for separator in separators:
            if not hasSeparator(triple_statement, separator, endpoint_19) and hasSeparator(triple_statement, separator, endpoint_23):
                  return True
    return False

   # Example usage
wd_pid = "http://www.wikidata.org/entity/P1006"
print(getConstraintSeparator(wd_pid))

stmt = "http://www.wikidata.org/entity/statement/Q12998-2619692C-2C47-4E03-A95B-518E02281017"
wasSeparatorValueAdded(wd_pid, stmt)

In [ ]:
df_single_val_repairs["SQ_added_property"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if pd.isna(row["SQ_added_property"]):
        if row['S_deleted'] is True:
            df_single_val_repairs.at[index, 'SQ_added_property'] = False
        elif row['no_separator'] is False and row['CQ_add_property_on_CQ_with_0'] is False:      
            result = wasSeparatorValueAdded(row['property'], row['wds1'])
            df_single_val_repairs.at[index, 'SQ_added_property'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.
(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["SQ_added_property"].value_counts()

In [ ]:
df_single_val_repairs[(df_single_val_repairs['SQ_added_property'] == None)]

In [ ]:
df_single_val_repairs.iloc[238692]

- In cases like index 238692, there were more than 1 statement with a separator with the same value and one of them got deleted, therefore we will test based on the separator value for a given instance

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1081")

In [ ]:
def getCountSeparatorValue(wds1, subject, wd_pid, pq_qua, endpoint):

    p_pid = wd_pid.replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    
    query = f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX wikibase: <http://wikiba.se/ontology#>
        PREFIX p: <http://www.wikidata.org/prop/>
        PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
        PREFIX ps: <http://www.wikidata.org/prop/statement/>
        PREFIX wd: <http://www.wikidata.org/entity/>
        PREFIX wdt: <http://www.wikidata.org/prop/direct/>

        SELECT (COUNT(?o) AS ?total)
        WHERE
        {{
            <{wds1}> <{pq_qua}> ?o.
            <{subject}> <{p_pid}>/<{pq_qua}> ?o
        }}
    """
    
    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find the 'literal' element containing the 'total' value
        literal_element = root.find('.//ns:literal', namespace)

        # Extract the text from the 'literal' element and convert to an integer
        total_value = int(literal_element.text) if literal_element is not None else 0
        
        return total_value
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
    

def checkIfComplementaryWithSeparatorWasRemoved(row):
    
    pq_qua_list = getConstraintSeparator(row['property'])
    if pq_qua_list is None:
        return False
    for pq_qua in pq_qua_list:
        count_2019 = getCountSeparatorValue(row['wds1'], row['subject'], row['property'], pq_qua, "ENTER_qEndpoint_WD_2019")
        count_2023 = getCountSeparatorValue(row['wds1'], row['subject'], row['property'], pq_qua, "ENTER_qEndpoint_WD_2023")
        if count_2023 < count_2019 and count_2023 == 1:
            return True
    return False

In [ ]:
checkIfComplementaryWithSeparatorWasRemoved(df_single_val_repairs.iloc[238692])

In [ ]:
df_single_val_repairs["Sc_deleted"].value_counts()

- and compute the deletions for Sc with separator:

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["Sc_deleted"] is False:
        if row['S_deleted'] is False and row['no_separator'] is False and row['CQ_add_property_on_CQ_with_0'] is False and row['SQ_added_property'] is False :      
            result = checkIfComplementaryWithSeparatorWasRemoved(row)
            df_single_val_repairs.at[index, 'Sc_deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

- listing non-classified repairs (yet):

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['CQ_add_property_on_CQ_with_0'] == False)
    (df_single_val_repairs['C_deleted'] == False)
    & (df_single_val_repairs['C_deprecated'] == False)
    & (df_single_val_repairs['CQ_added_exception'] == False)
    & (df_single_val_repairs['S_deleted'] == False) 
    & (df_single_val_repairs['Sc_deleted'] == False) #
    & (df_single_val_repairs['SQ_added_property'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240120]

In [ ]:
df_single_val_repairs.iloc[240120]['wds1']

- For index 240120, the instance didn't change but the constraint did, it had one separator and now it has 3, and the instance had that qualifier. So it's no longer a violation, it's a CQ_added_property (CQ+)
if S_deleted is False

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1082", "ENTER_qEndpoint_WD_2019")

In [ ]:
getConstraintSeparator("http://www.wikidata.org/entity/P1082", "ENTER_qEndpoint_WD_2023")

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getTripleSeparators(wds1, endpoint = "ENTER_qEndpoint_WD_2023"):
    
    # URL of the endpoint
    #endpoint = "ENTER_qEndpoint_WD_2023"

    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT
      ?p
    WHERE
    {{
       <{wds1}> ?p ?o.
      [] wikibase:qualifier ?p
    }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)
    
    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"
    
    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}
    
    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        if len(uris) == 0:
            return None
        
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None

In [ ]:
getTripleSeparators("http://www.wikidata.org/entity/statement/Q1000029-02ED7526-768E-47C1-BE19-F6BDB8110C07")

- starting to calculate T-box separator addition (CQ+):

In [ ]:
df_single_val_repairs["CQ_added_property_of_SQ_sep"] = None

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    df_single_val_repairs.at[index, 'CQ_added_property_of_SQ_sep'] = testTboxseparatorAdded(row)

    # Save a checkpoint every 10,000 rows
    if index % 50000 == 0:
        df_single_val_repairs.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
def testTboxseparatorAdded(row):
    if row["S_deleted"] is False:
        c_separators_2023 = getConstraintSeparator(row["property"], "ENTER_qEndpoint_WD_2023")
        if len(c_separators_2023) == 0:
            return False
        else:
            c_separators_2019 = getConstraintSeparator(row["property"], "ENTER_qEndpoint_WD_2019")
            if len(c_separators_2019) == 0:
                c_added_separators = c_separators_2023
            else:
                # A list of separators in 2023 who were not in 2019
                c_added_separators = [item for item in c_separators_2023 if item not in c_separators_2019]

            intance_separators = getTripleSeparators(row["wds1"])
            if intance_separators is None or len(intance_separators) == 0:
                return False
            else:
                for sep in intance_separators:
                    if sep in c_added_separators:
                        return True
    return False

In [ ]:
testTboxseparatorAdded(df_single_val_repairs.iloc[1])

In [ ]:
df_single_val_repairs["CQ_added_property_of_SQ_sep"].value_counts()

- checking non-classified rows (yet):

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['CQ_add_property_on_CQ_with_0'] == False)
    (df_single_val_repairs['C_deleted'] == False)
    & (df_single_val_repairs['C_deprecated'] == False)
    & (df_single_val_repairs['CQ_added_exception'] == False)
    & (df_single_val_repairs['S_deleted'] == False) 
    & (df_single_val_repairs['Sc_deleted'] == False) #
    & (df_single_val_repairs['SQ_added_property'] == False) 
    & (df_single_val_repairs['CQ_added_property_of_SQ_sep'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240163]

- For index 240163, it had 1 separator and more separators were added: P459, P518

In [ ]:
df_single_val_repairs.iloc[240163]['wds1']

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

- testing alternative context statement (Sc) deletion:

In [ ]:
def getStatementsList(row, endpoint = "ENTER_qEndpoint_WD_2023"):

    p_pid = row['property'].replace("http://www.wikidata.org/entity/", "http://www.wikidata.org/prop/")
    # SPARQL query
    query = f"""
    PREFIX wikibase: <http://wikiba.se/ontology#>
    SELECT ?o
        WHERE
        {{
          <{row['subject']}> <{p_pid}> ?o.
        }}
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/xhtml+xml,application/xml;"}

    response = requests.get(url,headers=headers)
    #print(response.text)
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        #print(response.text)
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        
        if len(uris) == 0:
            return []
        
        uris.remove(row['wds1'])
        return uris
    else:
        # If there's an error in the request, return None
        print("Error:", response.text)
        return None
    
def checkIfComplementaryWasRemoved(row):
    statement_list = getStatementsList(row, "ENTER_qEndpoint_WD_2019")
    for stmt in statement_list:
        if statementDeleted(stmt):
            return True
    return False

checkIfComplementaryWasRemoved(df_single_val_repairs.iloc[240163])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["Sc_deleted"] is False:
        if row['S_deleted'] is False and row['no_separator'] is False and row['SQ_added_property'] is False :      
            result = checkIfComplementaryWasRemoved(row)
            df_single_val_repairs.at[index, 'Sc_deleted'] = result

        # Save a checkpoint every 10,000 rows
        if index % 10000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")

# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["Sc_deleted"].value_counts()

- validating non-classified rows:

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['CQ_add_property_on_CQ_with_0'] == False)
    (df_single_val_repairs['C_deleted'] == False)
    & (df_single_val_repairs['C_deprecated'] == False)
    & (df_single_val_repairs['CQ_added_exception'] == False)
    & (df_single_val_repairs['S_deleted'] == False) 
    & (df_single_val_repairs['Sc_deleted'] == False) #
    & (df_single_val_repairs['SQ_added_property'] == False) 
    & (df_single_val_repairs['CQ_added_property_of_SQ_sep'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.iloc[240224]

- index 240224 represents context statement separator addition (SQc+):

In [ ]:
df_single_val_repairs.iloc[240224]['wds1']

In [ ]:
df_single_val_repairs['SQc_added_property'] = None

In [ ]:
def checkIfComplementaryHadSeparadorAdded(row):
    
    # URL of the endpoint
    endpoint_19 = "ENTER_qEndpoint_WD_2019"
    endpoint_23 = "ENTER_qEndpoint_WD_2023"
    
    separators = getConstraintSeparator(row['property'])
    stmts = getStatementsList(row)
        
    if separators and stmts:
        for stmt in stmts:
            for separator in separators:
                if not hasSeparator(stmt, separator, endpoint_19) and hasSeparator(stmt, separator, endpoint_23):
                      return True
    return False
    
checkIfComplementaryHadSeparadorAdded(df_single_val_repairs.iloc[240224])

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm
import os


# Process the rows with progress bar
for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    # Check for unprocessed rows
    if row["S_deleted"] is False:
        if row['no_separator'] is False:      
            result = checkIfComplementaryHadSeparadorAdded(row)
            df_single_val_repairs.at[index, 'SQc_added_property'] = result
        else:
            df_single_val_repairs.at[index, 'SQc_added_property'] = False
        
        # Save a checkpoint every 10,000 rows
        if index % 100000 == 0:
            df_single_val_repairs.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at row {index}.")
    else:
        df_single_val_repairs.at[index, 'SQc_added_property'] = False
        
            
# Save the final output
df_single_val_repairs.to_csv(checkpoint_file, index=False)
print("Processing complete. Final output saved.")

In [ ]:
df_single_val_repairs["SQc_added_property"].value_counts()

In [ ]:
df_single_val_repairs[
     #(df_single_val_repairs['no_separator'] == False)  
    # (df_single_val_repairs['CQ_add_property_on_CQ_with_0'] == False)
    (df_single_val_repairs['C_deleted'] == False)
    & (df_single_val_repairs['C_deprecated'] == False)
    & (df_single_val_repairs['CQ_added_exception'] == False)
    & (df_single_val_repairs['S_deleted'] == False) 
    & (df_single_val_repairs['Sc_deleted'] == False) #
    & (df_single_val_repairs['SQ_added_property'] == False) 
    & (df_single_val_repairs['CQ_added_property_of_SQ_sep'] == False) 
    & (df_single_val_repairs['SQc_added_property'] == False) 
     
    ]

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)

In [ ]:
# Select the two columns you want to print
print(df_single_val_repairs[['CQ_add_property_on_CQ_with_0', 'CQ_added_property_of_SQ_sep']])


In [ ]:
df_single_val_repairs.iloc[0]

In [ ]:
testTboxseparatorAdded(df_single_val_repairs.iloc[0])

In [ ]:
df_single_val_repairs.iloc[675336]

In [ ]:
df_single_val_repairs.iloc[675336]['wds1']

In [ ]:
df_single_val_repairs['CQ_added_property'] = df_single_val_repairs['CQ_add_property_on_CQ_with_0'] | df_single_val_repairs['CQ_added_property_of_SQ_sep']


In [ ]:
df_single_val_repairs['CQ_added_property'].value_counts()

- testing for context statement separators value replacement:

In [ ]:
import requests
import xml.etree.ElementTree as ET

def getAboxSeparatorValuesList(row, endpoint = "ENTER_qEndpoint_WD_2019"):

    pid = row['property'].replace("http://www.wikidata.org/entity/", "")
    # SPARQL query
    query = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX p: <http://www.wikidata.org/prop/>
    PREFIX ps: <http://www.wikidata.org/prop/statement/>
    PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX bd: <http://www.bigdata.com/rdf#>

    SELECT ?sep_val
    {{
      wd:{pid} p:P2302 ?CQ. 
      ?CQ ps:P2302 wd:Q19474404. 
      ?CQ pq:P4155/wikibase:qualifier ?sep_prop.

      <{row['subject']}> p:{pid} ?SQ. 
      ?SQ ?sep_prop ?sep_val.
    }} ORDER BY ASC(?sep_val)
    """

    # URL encode the query
    encoded_query = requests.utils.quote(query)

    # Build the complete URL
    url = f"{endpoint}?query={encoded_query}"

    # Send HTTP GET request
    headers = {"Accept": "application/sparql-results+xml"}

    response = requests.get(url, headers=headers)
    
    # Check if the request was successful and parse the response
    if response.ok:
        # Parse the XML response
        root = ET.fromstring(response.text)

        # Define the namespace used in the XML
        namespace = {'ns': 'http://www.w3.org/2005/sparql-results#'}

        # Find all 'uri' elements and extract their text
        uris = [uri_element.text for uri_element in root.findall('.//ns:uri', namespace)]

        return uris  # <- Just return everything found
    else:
        print("Error:", response.text)
        return None
    
print(getAboxSeparatorValuesList(df_single_val_repairs.iloc[675336]))
print(getAboxSeparatorValuesList(df_single_val_repairs.iloc[675336], "ENTER_qEndpoint_WD_2023"))

In [ ]:
from tqdm import tqdm
import os

df_single_val_repairs['SQ_separator_value_replaced'] = None

for index, row in tqdm(df_single_val_repairs.iterrows(), total=len(df_single_val_repairs)):
    
    if row['SQ_separator_value_replaced'] is None:
        if row["no_separator"] is True or row['S_deleted'] is True or row['Sc_deleted'] is True:
            df_single_val_repairs.at[index, 'SQ_separator_value_replaced'] = False
        else:
            separator_values_19 = getAboxSeparatorValuesList(row)
            if separator_values_19 is not None and len(separator_values_19) == 0:
                df_single_val_repairs.at[index, 'SQ_separator_value_replaced'] = False
            else:
                separator_values_23 = getAboxSeparatorValuesList(row, "ENTER_qEndpoint_WD_2023")
                if len(separator_values_23) == 0:
                    df_single_val_repairs.at[index, 'SQ_separator_value_replaced'] = False
                else:
                    for value in separator_values_19:
                        if value not in separator_values_23:
                            df_single_val_repairs.at[index, 'SQ_separator_value_replaced'] = True
                            break

In [ ]:
df_single_val_repairs['SQ_separator_value_replaced'].value_counts()

In [ ]:
df_single_val_repairs.to_csv("single_value_all_repairs.csv",index=False)